In [ ]:
from utils import (
    extract_scaling_metadata,
    predict_nuclei_labels,
    segment_organoids_from_cp_labels,
    map_df_column_to_labels,
)

from io_utils import (
    list_images, 
    ensure_output_dir, 
    load_precomputed_results_if_available
)
from pathlib import Path
import tifffile
import czifile
import napari
import numpy as np
import pandas as pd
import pyclesperanto_prototype as cle
import plotly.express as px

cle.select_device("RTX")

In [ ]:
# Copy the path where your .czi files are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY_WT = "./raw_data/20240607 PGE2MSN WT_organoids"
RAW_DATA_DIRECTORY_TUM = "./raw_data/20240703 PGE2MSN Tum_organoids"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# Minimum and maximum nuclei label volume to use for filtering predicted nuclei labels (min, max) i.e (250, 4000)
# If set to None, no filtering is applied and all predicted labels are returned
MIN_MAX_NUCLEI_VOLUME = (250, 4000)

# Cytoplasm thickness in voxels (starting from the nucleus)
# If padding is applied, the thickness will be applied to the padded nucleus
CYTOPLASM_THICKNESS = 2

# Padding to keep a safe distance between nuclei and cytoplasm borders
# It expands the nucleus by the padding value in all directions
NUCLEI_PADDING = 0

# List of markers used in the experiment, each with its corresponding channel index
MARKERS = [("UEA-1", 0), ("YAP", 1), ("Nuclei", 2), ("BCAT", 3)]

# Shared Napari colormap for nuclei_cyto_ratio overlays in every figure in figures_to_plot.
RATIO_COLORMAP = "inferno"

# How to set the global contrast range (ratio_vmin, ratio_vmax) shared across all figures.
# Change only this line to switch modes; companion constants below apply when relevant.
#
# A — global min/max: pool nuclei_cyto_ratio for every figures_to_plot entry (dropna), use min and max.
#     Full spread across the batch; extreme per-label ratios can compress mid-range contrast.
# B — global percentiles: same pooled ratios as A, use RATIO_COLORMAP_PERCENTILES as display limits.
#     Comparable across figures with less sensitivity to outliers than A.
# C — fixed limits: use RATIO_COLORMAP_FIXED_LIMITS; ignore pooled data for vmin/vmax (stable scale every run).
RATIO_COLORMAP_SCALE_MODE = "B"  # "A", "B", or "C"

# Used when RATIO_COLORMAP_SCALE_MODE == "B" (low, high) percentiles on pooled nuclei_cyto_ratio.
RATIO_COLORMAP_PERCENTILES = (1, 99)

# Used when RATIO_COLORMAP_SCALE_MODE == "C"; replace with biologically meaningful bounds.
RATIO_COLORMAP_FIXED_LIMITS = (0.0, 3.0)

In [ ]:
# Construct per_label_results_csv_path for both WT and Tumor directories
PER_LABEL_RESULTS_CSV_PATH_WT = f"./results/bp_results/{Path(RAW_DATA_DIRECTORY_WT).stem}/per_label_results_{Path(RAW_DATA_DIRECTORY_WT).stem}_MIN_None_MAX_None_CT{CYTOPLASM_THICKNESS}_NP{NUCLEI_PADDING}.csv"
PER_LABEL_RESULTS_CSV_PATH_TUM = f"./results/bp_results/{Path(RAW_DATA_DIRECTORY_TUM).stem}/per_label_results_{Path(RAW_DATA_DIRECTORY_TUM).stem}_MIN_None_MAX_None_CT{CYTOPLASM_THICKNESS}_NP{NUCLEI_PADDING}.csv"

df_wt = pd.read_csv(PER_LABEL_RESULTS_CSV_PATH_WT)
df_tum = pd.read_csv(PER_LABEL_RESULTS_CSV_PATH_TUM)
merged_df_all = pd.concat([df_wt, df_tum], ignore_index=True)

In [ ]:
figures_to_plot = ["RY20240607 4541 MSN 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 3",
                    "RY20240608 4543 BCM 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 1",
                    "RY20240703 4312 BCM 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 4.1",
                    "RY20240703 4312 MSN 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 3.1",
                    "RY20240611 4541 BCM_60min 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 2",
                    "RY20240611 4541 BCM_60min 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 3",
                    "RY20240611 4541 BCM_60min 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 4",
                    "RY20240613 4543 MSN_60min 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 6",
                    "RY20240725 4385 BCM_60min 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 4.1",
                    "RY20240725 4385 MSN_60min 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 4.1",
                    "RY20240703 4312 MSN 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 4.1",
                    "RY20240703 4312 MSN 20X Hoechst bcat_GaMAlexa555 YAP_GaRAlexa647 UEA1_FITC 5.1"]

In [ ]:
images = []
for raw_data_dir in [RAW_DATA_DIRECTORY_WT, RAW_DATA_DIRECTORY_TUM]:
    images.extend(list_images(Path(raw_data_dir), "czi"))

nuclei_labels_dir_wt = ensure_output_dir(
    RAW_DATA_DIRECTORY_WT, Path(RAW_DATA_DIRECTORY_WT).stem, results_type="nuclei_labels"
)
nuclei_labels_dir_tum = ensure_output_dir(
    RAW_DATA_DIRECTORY_TUM, Path(RAW_DATA_DIRECTORY_TUM).stem, results_type="nuclei_labels"
)


def parse_filename_metadata(stem: str) -> dict:
    parts = stem.split(" ")
    return {
        "experiment_id": parts[0],
        "mouse_id": parts[1],
        "treatment_id": parts[2],
        "replica_id": parts[-1],
    }


def normalize_metadata_value(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip()
    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
        return format(number, "g")
    except ValueError:
        return text


def filter_merged_by_metadata(df: pd.DataFrame, meta: dict) -> pd.DataFrame:
    mask = pd.Series(True, index=df.index)
    for column in ("experiment_id", "mouse_id", "treatment_id", "replica_id"):
        mask &= df[column].map(normalize_metadata_value) == normalize_metadata_value(meta[column])
    if "cyto_thickness" in df.columns:
        mask &= df["cyto_thickness"] == CYTOPLASM_THICKNESS
    if "nuclei_padding" in df.columns:
        mask &= df["nuclei_padding"] == NUCLEI_PADDING
    return df.loc[mask].copy()


def pool_nuclei_cyto_ratios(df: pd.DataFrame, figure_stems: list[str]) -> pd.Series:
    ratio_chunks = []
    for figure_stem in figure_stems:
        image_df = filter_merged_by_metadata(df, parse_filename_metadata(figure_stem))
        ratio_chunks.append(image_df["nuclei_cyto_ratio"].dropna())
    if not ratio_chunks:
        return pd.Series(dtype=float)
    return pd.concat(ratio_chunks, ignore_index=True)


def resolve_ratio_contrast_limits(
    df: pd.DataFrame,
    figure_stems: list[str],
    mode: str,
    percentiles: tuple[float, float],
    fixed_limits: tuple[float, float],
) -> tuple[float, float]:
    mode = mode.upper()
    if mode == "C":
        ratio_vmin, ratio_vmax = fixed_limits
    else:
        pooled_values = pool_nuclei_cyto_ratios(df, figure_stems)
        if pooled_values.empty:
            missing = [parse_filename_metadata(stem) for stem in figure_stems]
            raise ValueError(
                "No nuclei_cyto_ratio values found for figures_to_plot with metadata: "
                f"{missing}"
            )
        if mode == "A":
            ratio_vmin = float(pooled_values.min())
            ratio_vmax = float(pooled_values.max())
        elif mode == "B":
            ratio_vmin, ratio_vmax = np.percentile(pooled_values, percentiles)
            ratio_vmin = float(ratio_vmin)
            ratio_vmax = float(ratio_vmax)
        else:
            raise ValueError(f"Unsupported RATIO_COLORMAP_SCALE_MODE: {mode!r}")

    if ratio_vmin == ratio_vmax:
        ratio_vmax = ratio_vmin + np.finfo(float).eps

    return ratio_vmin, ratio_vmax


def find_image_path(search_filename: str, image_paths: list[str]) -> str:
    for image_path in image_paths:
        if search_filename in image_path:
            return image_path
    raise FileNotFoundError(f"Could not find a .czi file matching: {search_filename}")


def resolve_nuclei_labels_dir(file_path: Path) -> Path:
    resolved_image = file_path.resolve()
    wt_root = Path(RAW_DATA_DIRECTORY_WT).resolve()
    tum_root = Path(RAW_DATA_DIRECTORY_TUM).resolve()
    if resolved_image.is_relative_to(wt_root):
        return nuclei_labels_dir_wt
    if resolved_image.is_relative_to(tum_root):
        return nuclei_labels_dir_tum
    raise ValueError(
        f"File path {file_path} does not match WT or TUM raw data directories."
    )


FIGURES_OUTPUT_DIR = Path("results/figures_to_plot")


def sanitize_export_filename(name: str) -> str:
    sanitized = name.strip()
    for char in '<>:"/\\|?*':
        sanitized = sanitized.replace(char, "_")
    return sanitized


def export_viewer_layer_png(viewer, layer_name: str, output_dir: Path = FIGURES_OUTPUT_DIR) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{sanitize_export_filename(layer_name)}.png"
    visible_state = {layer.name: layer.visible for layer in viewer.layers}
    try:
        for layer in viewer.layers:
            layer.visible = layer.name == layer_name
        viewer.screenshot(path=str(output_path), canvas_only=True)
    finally:
        for layer in viewer.layers:
            layer.visible = visible_state[layer.name]
    return output_path

In [ ]:
ratio_vmin, ratio_vmax = resolve_ratio_contrast_limits(
    merged_df_all,
    figures_to_plot,
    RATIO_COLORMAP_SCALE_MODE,
    RATIO_COLORMAP_PERCENTILES,
    RATIO_COLORMAP_FIXED_LIMITS,
)
print(
    f"Shared nuclei_cyto_ratio contrast limits ({RATIO_COLORMAP_SCALE_MODE}): "
    f"({ratio_vmin}, {ratio_vmax})"
)

In [ ]:
for figure_stem in figures_to_plot:
    image = find_image_path(figure_stem, images)
    file_path = Path(image)
    filename = file_path.stem
    meta = parse_filename_metadata(filename)

    image_df = filter_merged_by_metadata(merged_df_all, meta)
    if image_df.empty:
        raise ValueError(f"No per-label rows matched metadata for {figure_stem}: {meta}")

    nuclei_labels_dir = resolve_nuclei_labels_dir(file_path)

    img = czifile.imread(image)
    img = img.squeeze()

    viewer = napari.Viewer(ndisplay=3)
    viewer.add_image(
        img,
        channel_axis=0,
        colormap=["white", "magenta", "cyan", "yellow"],
        name=[marker[0] for marker in MARKERS],
    )

    scaling_x_um, scaling_y_um, scaling_z_um = extract_scaling_metadata(file_path)
    rescale_factor = scaling_z_um / np.mean([scaling_x_um, scaling_y_um])

    labels_layer_name = f"nuclei_labels_{figure_stem}"
    ratio_layer_name = f"nuclei_cyto_ratio_{figure_stem}"

    nuclei_labels = load_precomputed_results_if_available(
        nuclei_labels_dir, filename, results_type="nuclei_labels"
    )

    if nuclei_labels is not None:
        print(f"Predictions already calculated for: {filename} ...loading")
        if MIN_MAX_NUCLEI_VOLUME is not None:
            from utils import _keep_objects_in_size_range

            nuclei_labels = _keep_objects_in_size_range(nuclei_labels, MIN_MAX_NUCLEI_VOLUME)
    else:
        nuclei_labels = predict_nuclei_labels(
            img,
            rescale_factor,
            NUCLEI_CHANNEL,
            MIN_MAX_NUCLEI_VOLUME,
            visualize=False,
        )
        nuclei_labels_path = nuclei_labels_dir / f"{filename}_nuclei_labels.tif"
        tifffile.imwrite(nuclei_labels_path, nuclei_labels)

    organoid_labels = segment_organoids_from_cp_labels(nuclei_labels, cytoplasm_thickness=1)
    organoid_mask = organoid_labels > 0
    nuclei_labels_filtered = nuclei_labels.copy()
    nuclei_labels_filtered[~organoid_mask] = 0

    viewer.add_labels(nuclei_labels_filtered, name=labels_layer_name)

    if "organoid_id" in image_df.columns:
        image_df = image_df.loc[image_df["organoid_id"] > 0].copy()
    if image_df.empty:
        raise ValueError(
            f"No per-label rows with organoid_id > 0 matched metadata for {figure_stem}: {meta}"
        )

    map_df_column_to_labels(
        nuclei_labels_filtered,
        image_df,
        value_column="nuclei_cyto_ratio",
        colormap=RATIO_COLORMAP,
        contrast_limits=(ratio_vmin, ratio_vmax),
        visualize=True,
        viewer=viewer,
        layer_name=ratio_layer_name,
    )

    labels_png_path = export_viewer_layer_png(viewer, labels_layer_name)
    ratio_png_path = export_viewer_layer_png(viewer, ratio_layer_name)
    print(f"Saved {labels_png_path}")
    print(f"Saved {ratio_png_path}")
    viewer.close()